Demo — Reinforcement Learning para Trading
¶
Aula 7 | AI Engineering | MBA


Vamos formular Algorithmic Trading como 
MDP
 e treinar um agente 
Q-Learning tabular
 sobre dados sintéticos de FICT3.




⚠ Objetivo pedagógico: entender a 
formulação
 do problema. Q-Learning tabular é didático mas insuficiente para mercados reais.

1. Setup e dados
¶

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 7
np.random.seed(SEED)

df = pd.read_csv("ohlcv_ativo.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
df["ret"] = df["close"].pct_change().fillna(0)
print(f"Periodo: {df['date'].min().date()} -> {df['date'].max().date()} | {len(df)} dias")
df.head()


Esse conjunto de dados é de 750 pregoes de um ativo fictício chamado FICT3 gerado via Movimento Browniano Geométrico. 

O FICT3 é puro ruído gaussiano. Nao tem padrao real, reversao à média e nem momentum verdadeiro. O objetivo é puramente didático. Na prática, se o agente encontrar padroes, é overfit. 

estamos analisando o retorno por conta do MDP (requer estacionariedade - média e variancia estáveis no tempo). 

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(df["date"], df["close"], color="#0A2540")
axes[0].set_title("Preco de fechamento - FICT3"); axes[0].set_ylabel("R$")
axes[1].bar(df["date"], df["volume"], color="#000000", width=0.2)
axes[1].set_title("Volume diario"); axes[1].set_ylabel("acoes")
plt.tight_layout(); plt.show()


2. Ambiente: Estado, Ação, Recompensa
¶


Estado
 $s$: quintil da média móvel 5d dos retornos (5 estados discretos)


Ação
 $a$: 0 = flat, 1 = long, 2 = short


Recompensa
 $r$: retorno do dia seguinte com sinal conforme a ação


Horizonte
: 1 dia (decisão diária)

In [ ]:
df["mom5"] = df["ret"].rolling(5).mean() #retorno médio dos últimos 5 dias
df["state"] = pd.qcut(df["mom5"], q=5, labels=False)
df = df.dropna().reset_index(drop=True)
df["state"] = df["state"].astype(int)

fig, ax = plt.subplots(figsize=(7, 3.5))
df["state"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#0A2540")
ax.set_title("Distribuicao dos estados (quintis de momentum 5d)")
ax.set_xlabel("Estado"); ax.set_ylabel("# observacoes")
plt.tight_layout(); plt.show()
#estados: 0 - forte queda; 1 - queda leve; 2 - lateral; 3 - alta leve; 4 - forte alta

3. Q-Learning Tabular
¶

In [ ]:
n_states  = 5
n_actions = 3   # 0=flat, 1=long, 2=short
alpha     = 0.10
gamma     = 0.95
eps_start, eps_end = 0.30, 0.01 # comeca com 30% de eploracao e decai até 1%
n_episodes = 300

split = int(0.7 * len(df))
train = df.iloc[:split].reset_index(drop=True)
test  = df.iloc[split:].reset_index(drop=True)

def reward(action, ret_next):
    if action == 1: return ret_next    # long: ganha se subir
    if action == 2: return -ret_next   # short: ganha se cair
    return 0.0                         # flat: nao joga

Q = np.zeros((n_states, n_actions))
returns_per_episode = []

for ep in range(n_episodes):
    eps = eps_end + (eps_start - eps_end) * (1 - ep / n_episodes)
    ep_reward = 0.0
    for i in range(len(train) - 1):
        s      = int(train["state"].iat[i]) #iat: similar ao iloc - acessar um único valor em um DF
        s_next = int(train["state"].iat[i+1])
        r_next = float(train["ret"].iat[i+1])

        if np.random.rand() < eps:
            a = np.random.randint(n_actions)
        else:
            a = int(np.argmax(Q[s]))

        r = reward(a, r_next)
        Q[s, a] += alpha * (r + gamma * Q[s_next].max() - Q[s, a])
        ep_reward += r
    returns_per_episode.append(ep_reward)

print("Q-table final:")
print(np.round(Q, 4))


Detalhando a equacao de Bellman:

* $\alpha$: taxa de aprendizado - o quanto a nova info atualiza a tabela
* $\gamma$: fator de desconto - quanto valorizo recompensas futuras
* $r$: recompensa imediada observada (d+1)
* $\max_{a'}Q(s',a')$: melhor valor esperado a partir do próximo estado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(Q, annot=True, fmt=".3f", cmap="RdYlGn", center=0,
            xticklabels=["flat","long","short"],
            yticklabels=[f"s{i}" for i in range(n_states)], ax=axes[0])
axes[0].set_title("Q-table aprendida")
axes[1].plot(pd.Series(returns_per_episode).rolling(20).mean(), color="#0A2540")
axes[1].set_title("Recompensa acumulada / episodio (MM20)")
axes[1].set_xlabel("Episodio"); axes[1].set_ylabel("Soma de recompensas")
plt.tight_layout(); plt.show()


4. Avaliação Out-of-Sample
¶

In [ ]:
test = test.copy()
test["action"] = test["state"].map(lambda s: int(np.argmax(Q[int(s)]))) # o agente faz o que aprendeu, sem explorar
test["strategy_ret"] = test.apply(lambda r: reward(int(r["action"]), float(r["ret"])), axis=1)
test["equity_rl"] = (1 + test["strategy_ret"]).cumprod()
test["equity_bh"] = (1 + test["ret"]).cumprod()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test["date"], test["equity_rl"], label="Q-Learning", color="#0A2540", linewidth=2)
ax.plot(test["date"], test["equity_bh"], label="Buy & Hold", color="#1FB8CD", linewidth=2)
ax.set_title("Equity curve - Out-of-Sample")
ax.set_ylabel("Equity (1 = base)"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
def sharpe(returns, periods=252):
    mu = returns.mean() * periods
    sd = returns.std() * (periods ** 0.5)
    return mu / sd if sd > 0 else 0.0

def max_drawdown(equity):
    peak = equity.cummax()
    dd = (equity / peak) - 1
    return dd.min()

metrics = pd.DataFrame({
    "Q-Learning": [sharpe(test["strategy_ret"]), max_drawdown(test["equity_rl"]), test["equity_rl"].iloc[-1] - 1],
    "Buy & Hold": [sharpe(test["ret"]),          max_drawdown(test["equity_bh"]), test["equity_bh"].iloc[-1] - 1],
}, index=["Sharpe (anual)", "Max Drawdown", "Retorno total"])
print(metrics.round(3))


* sharpe anualizado = (retorno medio anualizado)/(volatilidade anualizada)
* Max Drawndown = pior queda acumulada do pico ao vale. 

5. Análise de Sensibilidade
¶

In [ ]:
alphas = [0.05, 0.10, 0.20]
gammas = [0.80, 0.90, 0.99]
grid   = np.zeros((len(alphas), len(gammas)))

for ia, a_lr in enumerate(alphas):
    for ig, g_df in enumerate(gammas):
        Q2 = np.zeros((n_states, n_actions))
        for ep in range(150):
            eps = 0.1
            for i in range(len(train) - 1):
                s   = int(train["state"].iat[i])
                s_n = int(train["state"].iat[i+1])
                r_n = float(train["ret"].iat[i+1])
                if np.random.rand() < eps:
                    a = np.random.randint(n_actions)
                else:
                    a = int(np.argmax(Q2[s]))
                r = reward(a, r_n)
                Q2[s, a] += a_lr * (r + g_df * Q2[s_n].max() - Q2[s, a])
        acts = test["state"].map(lambda s: int(np.argmax(Q2[int(s)])))
        rets = [reward(int(a), float(r)) for a, r in zip(acts, test["ret"])]
        grid[ia, ig] = sharpe(pd.Series(rets))

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(grid, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            xticklabels=gammas, yticklabels=alphas, ax=ax)
ax.set_xlabel("gamma"); ax.set_ylabel("alpha")
ax.set_title("Sharpe out-of-sample por (alpha, gamma)")
plt.tight_layout(); plt.show()


Em dados reais:

* $\alpha$ muito alto: instável
* $\alpha$ muito baixo: nao converge
* $\gamma$ alto: boa visao de longo prazo
* $\gamma$ baixo: míope

6. Exercícios propostos



Custos
Reescreva a funcao reward e subtraia 0.001 do reward em mudança de ação. Faca as alteracoes necessárias no treinamento


In [ ]:
#resposta

In [ ]:
#resposta